# Q1 - Hadoop HDFS and PySpark Analytics
## Big Data Analytical Methods - CA2
### Student: Arkadiusz Jedrzejewski,  sba25006

This notebook demonstrates distributed data processing using Apache Spark (PySpark).
The dataset used is the Global Weather Repository, containing weather observations
from countries worldwide, stored in Hadoop Distributed File System (HDFS).

## Step 1: Initialize Spark Session

In this step, I am setting up a SparkSession which is required to work with PySpark.
I named the application "WeatherAnalytics" to identify it in the Spark UI.
The master is set to local[*] which means Spark will use all available CPU cores
on my machine for processing.

In [1]:
#Create a Spark session
from pyspark.sql import SparkSession

In [2]:
# Import the os module to set environment variables
import os

# Set Java home path required for PySpark to work on Windows
os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-11.0.31.11-hotspot"

from pyspark.sql import SparkSession

# Create Spark Session for Weather Analytics
spark = SparkSession.builder \
    .appName("WeatherAnalytics") \
    .master("local[*]") \
    .getOrCreate()


In [3]:
print("Spark Ver.:", spark.version)
print("Spark Session created")

Spark Ver.: 3.5.1
Spark Session created


## Step 2: Load Dataset from HDFS

In this step, I am loading the Global Weather Repository CSV file from HDFS 
where I stored it earlier in the /CA2_sba25006 folder.
The dataset contains weather data from countries around the world such as 
temperature, humidity, wind speed and air quality readings.
I used inferSchema=True so that Spark can automatically detect the correct 
data types for each column instead of me having to define them manually.

In [5]:
# Load the dataset from local file system
# Note: Dataset was uploaded to HDFS at /CA2_sba25006/GlobalWeatherRepository.csv
# Loading locally for PySpark processing on Windows environment
df = spark.read.csv(
    r"C:\Users\arkje\OneDrive\Documents\GIT\big-data-weather-analytics\DATASETS\GlobalWeatherRepository.csv",
    header=True,
    inferSchema=True
)

# Show basic information about the dataset
print("Total number of rows:", df.count())
print("Total number of columns:", len(df.columns))
df.printSchema()

Total number of rows: 148515
Total number of columns: 41
root
 |-- country: string (nullable = true)
 |-- location_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- last_updated_epoch: integer (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- temperature_celsius: double (nullable = true)
 |-- temperature_fahrenheit: double (nullable = true)
 |-- condition_text: string (nullable = true)
 |-- wind_mph: double (nullable = true)
 |-- wind_kph: double (nullable = true)
 |-- wind_degree: integer (nullable = true)
 |-- wind_direction: string (nullable = true)
 |-- pressure_mb: double (nullable = true)
 |-- pressure_in: double (nullable = true)
 |-- precip_mm: double (nullable = true)
 |-- precip_in: double (nullable = true)
 |-- humidity: integer (nullable = true)
 |-- cloud: integer (nullable = true)
 |-- feels_like_celsius: double (nullable = true)
 |-- feels_like_f

## Step 3: Data Cleaning and Preprocessing

Before performing any analysis, I need to clean and prepare the dataset.
In this step I am removing any duplicate records and dropping rows with 
null values in the most important columns such as temperature, humidity 
and country. I am also selecting only the columns that are relevant 
for my analysis to make the data easier to work with.

In [5]:
# Check for missing values in key columns
print("Missing values per column:")
from pyspark.sql.functions import col, sum as spark_sum

missing = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
missing.show()

Missing values per column:


NameError: name 'df' is not defined